# Just execute below!!!

In [2]:
import tkinter as tk
from tkinter import ttk, messagebox
from tkinter.scrolledtext import ScrolledText

import numpy as np
import ast
import time
import io
from contextlib import redirect_stdout

import RandomTilings as RT

import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.figure import Figure


root = tk.Tk()
root.title("Random Tiling Studio")
root.geometry("900x650")


# Style
style = ttk.Style(root)

style.configure(
    "Active.TLabelframe.Label",
    foreground="black"
)

style.configure(
    "Disabled.TLabelframe.Label",
    foreground="gray"
)


# Overall layout: left controls, right side
frame_input = ttk.Frame(root, padding=10)
frame_input.grid(row=0, column=0, sticky="ns")

frame_right = ttk.Frame(root)
frame_right.grid(row=0, column=1, sticky="nsew")

# Right side: plot on top, log below
frame_plot = ttk.Frame(frame_right)
frame_plot.grid(row=0, column=0, sticky="nsew")

frame_log = ttk.LabelFrame(frame_right, text="Log")
frame_log.grid(row=1, column=0, sticky="ew", padx=8, pady=(0, 8))

root.grid_rowconfigure(0, weight=1)
root.grid_columnconfigure(0, weight=0)
root.grid_columnconfigure(1, weight=1)

frame_right.grid_rowconfigure(0, weight=1)
frame_right.grid_rowconfigure(1, weight=0)
frame_right.grid_columnconfigure(0, weight=1)


# Log window
text_log = ScrolledText(
    frame_log,
    height=6,
    state="disabled",
    wrap="word"
)
text_log.grid(row=0, column=0, sticky="ew", padx=8, pady=(8, 4))

button_clear_log = ttk.Button(
    frame_log,
    text="Clear Log"
)
button_clear_log.grid(row=1, column=0, sticky="e", padx=8, pady=(0, 8))

frame_log.grid_columnconfigure(0, weight=1)


def write_log(message, level="INFO"):
    text_log.config(state="normal")
    text_log.insert("end", f"[{level}] {message}\n")
    text_log.see("end")
    text_log.config(state="disabled")


def clear_log():
    text_log.config(state="normal")
    text_log.delete("1.0", "end")
    text_log.config(state="disabled")


button_clear_log.config(command=clear_log)


# Input variables
var_type = tk.StringVar(value="Aztec")
var_n = tk.IntVar(value=1)

# Hexagon side scaling variables
var_a = tk.DoubleVar(value=1.0)
var_b = tk.DoubleVar(value=1.0)
var_c = tk.DoubleVar(value=1.0)

# Plot variables
var_edge = tk.DoubleVar(value=0.0)
var_paths = tk.BooleanVar(value=False)
var_dots = tk.BooleanVar(value=False)
var_coloring = tk.StringVar(value="standard")

# Store actual weight matrix here
weight_matrix = np.array([[1]], dtype=float)

# Store current random tiling model here
RandTiling = None

# Store current Matplotlib canvas/toolbar/figure here
canvas = None
toolbar = None
current_fig = None


#### Model Frame
frame_model = ttk.LabelFrame(frame_input, text="Model")
frame_model.grid(row=0, column=0, sticky="ew", padx=8, pady=8)


##### Type Input
ttk.Label(frame_model, text="Type").grid(
    row=0,
    column=0,
    sticky="nw",
    padx=(8, 8),
    pady=4
)

frame_type = ttk.Frame(frame_model)
frame_type.grid(row=0, column=1, sticky="w", padx=(0, 8), pady=4)


#### Size Input
ttk.Label(frame_model, text="Size").grid(
    row=1,
    column=0,
    sticky="w",
    padx=(8, 8),
    pady=4
)

entry_n = ttk.Entry(
    frame_model,
    textvariable=var_n,
    width=10
)
entry_n.grid(row=1, column=1, sticky="ew", padx=(0, 8), pady=4)


#### Weight Input
ttk.Label(frame_model, text="Weight").grid(
    row=2,
    column=0,
    sticky="w",
    padx=(8, 8),
    pady=4
)

button_weight_edit = ttk.Button(
    frame_model,
    text="Edit"
)
button_weight_edit.grid(row=2, column=1, sticky="ew", padx=(0, 8), pady=4)


#### Hexagon Input
frame_hexagon = ttk.LabelFrame(
    frame_model,
    text="Hexagon scaling",
    style="Active.TLabelframe"
)
frame_hexagon.grid(
    row=3,
    column=0,
    columnspan=2,
    sticky="ew",
    padx=8,
    pady=(4, 8)
)

label_a = ttk.Label(frame_hexagon, text="a")
label_a.grid(row=0, column=0, sticky="w", padx=(8, 8), pady=4)

entry_a = ttk.Entry(frame_hexagon, textvariable=var_a, width=10)
entry_a.grid(row=0, column=1, sticky="ew", padx=(0, 8), pady=4)

label_b = ttk.Label(frame_hexagon, text="b")
label_b.grid(row=1, column=0, sticky="w", padx=(8, 8), pady=4)

entry_b = ttk.Entry(frame_hexagon, textvariable=var_b, width=10)
entry_b.grid(row=1, column=1, sticky="ew", padx=(0, 8), pady=4)

label_c = ttk.Label(frame_hexagon, text="c")
label_c.grid(row=2, column=0, sticky="w", padx=(8, 8), pady=4)

entry_c = ttk.Entry(frame_hexagon, textvariable=var_c, width=10)
entry_c.grid(row=2, column=1, sticky="ew", padx=(0, 8), pady=4)

frame_hexagon.grid_columnconfigure(1, weight=1)


def update_coloring_options():
    if "combo_coloring" not in globals():
        return

    if var_type.get() == "Aztec":
        values = ["standard", "alternative", "gray", "aztec gray"]
    else:
        values = ["standard", "alternative", "gray"]

    combo_coloring.config(values=values)

    if var_coloring.get() not in values:
        var_coloring.set("standard")


def update_type_inputs():
    is_hexagon = var_type.get() == "Hexagon"
    state = "normal" if is_hexagon else "disabled"

    label_a.config(state=state)
    label_b.config(state=state)
    label_c.config(state=state)

    entry_a.config(state=state)
    entry_b.config(state=state)
    entry_c.config(state=state)

    if is_hexagon:
        frame_hexagon.configure(style="Active.TLabelframe")
    else:
        frame_hexagon.configure(style="Disabled.TLabelframe")

    update_coloring_options()


radio_aztec = ttk.Radiobutton(
    frame_type,
    text="Aztec",
    variable=var_type,
    value="Aztec",
    command=update_type_inputs
)
radio_aztec.grid(row=0, column=0, sticky="w")

radio_hexagon = ttk.Radiobutton(
    frame_type,
    text="Hexagon",
    variable=var_type,
    value="Hexagon",
    command=update_type_inputs
)
radio_hexagon.grid(row=1, column=0, sticky="w")


def open_weight_editor():
    global weight_matrix

    window = tk.Toplevel(root)
    window.title("Edit weight matrix")
    window.geometry("500x300")

    ttk.Label(
        window,
        text="Enter a numeric matrix, e.g. [[1], [1]]"
    ).grid(row=0, column=0, sticky="w", padx=10, pady=(10, 4))

    text_weight = ScrolledText(window, height=8, width=50)
    text_weight.grid(row=1, column=0, sticky="nsew", padx=10, pady=4)

    text_weight.insert("1.0", repr(weight_matrix.tolist()))

    def apply_weight():
        global weight_matrix

        raw = text_weight.get("1.0", "end").strip()

        try:
            value = ast.literal_eval(raw)
            arr = np.array(value, dtype=float)

            if arr.ndim != 2:
                raise ValueError("Weight must be a 2D matrix.")

            if arr.size == 0:
                raise ValueError("Weight matrix must not be empty.")

            weight_matrix = arr
            write_log(f"Updated weight matrix to shape {arr.shape}.")
            window.destroy()

        except Exception as e:
            messagebox.showerror(
                "Invalid weight matrix",
                f"Could not parse weight matrix:\n\n{e}"
            )

    frame_buttons = ttk.Frame(window)
    frame_buttons.grid(row=2, column=0, sticky="ew", padx=10, pady=10)

    button_cancel = ttk.Button(
        frame_buttons,
        text="Cancel",
        command=window.destroy
    )
    button_cancel.grid(row=0, column=0, sticky="e", padx=4)

    button_apply = ttk.Button(
        frame_buttons,
        text="Apply",
        command=apply_weight
    )
    button_apply.grid(row=0, column=1, sticky="e", padx=4)

    frame_buttons.grid_columnconfigure(0, weight=1)

    window.grid_rowconfigure(1, weight=1)
    window.grid_columnconfigure(0, weight=1)


button_weight_edit.config(command=open_weight_editor)


def create_model():
    global RandTiling

    try:
        tiling_type = var_type.get()
        n = var_n.get()
        w = weight_matrix

        if n <= 0:
            raise ValueError("Size must be positive.")

        buffer = io.StringIO()
        start = time.perf_counter()

        with redirect_stdout(buffer):
            if tiling_type == "Aztec":
                RandTiling = RT.Aztec(n, w)

            elif tiling_type == "Hexagon":
                a = var_a.get()
                b = var_b.get()
                c = var_c.get()

                RandTiling = RT.Hexagon(n, w, a, b, c)

            else:
                raise ValueError(f"Unknown tiling type: {tiling_type}")

        elapsed = time.perf_counter() - start
        printed_output = buffer.getvalue().strip()

        write_log(f"Created {tiling_type} model in {elapsed:.3f}s.")

        if printed_output:
            write_log(printed_output, level="WARNING")

    except Exception as e:
        write_log(f"Could not create model: {e}", level="ERROR")


def shuffle_model():
    global RandTiling

    try:
        if RandTiling is None:
            raise ValueError("No model has been created yet.")

        buffer = io.StringIO()
        start = time.perf_counter()

        with redirect_stdout(buffer):
            RandTiling.shuffle()

        elapsed = time.perf_counter() - start
        printed_output = buffer.getvalue().strip()

        write_log(f"Shuffled model in {elapsed:.3f}s.")

        if printed_output:
            write_log(printed_output, level="WARNING")

    except Exception as e:
        write_log(f"Could not shuffle model: {e}", level="ERROR")


button_create_model = ttk.Button(
    frame_model,
    text="Create Model",
    command=create_model
)
button_create_model.grid(
    row=4,
    column=0,
    columnspan=2,
    sticky="ew",
    padx=8,
    pady=(4, 4)
)


button_shuffle_model = ttk.Button(
    frame_model,
    text="Shuffle",
    command=shuffle_model
)
button_shuffle_model.grid(
    row=5,
    column=0,
    columnspan=2,
    sticky="ew",
    padx=8,
    pady=(4, 8)
)


#### Plot Options Frame
frame_plot_options = ttk.LabelFrame(frame_input, text="Plot")
frame_plot_options.grid(row=1, column=0, sticky="ew", padx=8, pady=8)


#### Edge Input
ttk.Label(frame_plot_options, text="Edge width").grid(
    row=0,
    column=0,
    sticky="w",
    padx=(8, 8),
    pady=4
)

entry_edge = ttk.Entry(
    frame_plot_options,
    textvariable=var_edge,
    width=10
)
entry_edge.grid(row=0, column=1, sticky="ew", padx=(0, 8), pady=4)


def update_plot_inputs():
    if var_paths.get():
        check_dots.config(state="normal")
    else:
        check_dots.config(state="disabled")
        var_dots.set(False)


check_paths = ttk.Checkbutton(
    frame_plot_options,
    text="Show paths",
    variable=var_paths,
    command=update_plot_inputs
)
check_paths.grid(
    row=1,
    column=0,
    columnspan=2,
    sticky="w",
    padx=8,
    pady=4
)


check_dots = ttk.Checkbutton(
    frame_plot_options,
    text="Show dots",
    variable=var_dots
)
check_dots.grid(
    row=2,
    column=0,
    columnspan=2,
    sticky="w",
    padx=8,
    pady=4
)


ttk.Label(frame_plot_options, text="Coloring").grid(
    row=3,
    column=0,
    sticky="w",
    padx=(8, 8),
    pady=4
)

combo_coloring = ttk.Combobox(
    frame_plot_options,
    textvariable=var_coloring,
    values=["standard", "alternative", "gray", "aztec gray"],
    state="readonly",
    width=14
)
combo_coloring.grid(row=3, column=1, sticky="ew", padx=(0, 8), pady=4)

frame_plot_options.grid_columnconfigure(1, weight=1)


def display_figure(fig):
    """
    Display a Matplotlib figure inside frame_plot on the right.

    This function also closes figures from pyplot's perspective,
    so Jupyter does not display/store them as notebook output.
    """
    global canvas, toolbar, current_fig

    # Close the previously displayed figure from pyplot's perspective.
    if current_fig is not None:
        plt.close(current_fig)
        current_fig = None

    # Remove old toolbar and canvas from Tk.
    if toolbar is not None:
        toolbar.destroy()
        toolbar = None

    if canvas is not None:
        canvas.get_tk_widget().destroy()
        canvas = None

    # Embed new figure in Tk.
    canvas = FigureCanvasTkAgg(fig, master=frame_plot)
    canvas.draw()

    toolbar = NavigationToolbar2Tk(canvas, frame_plot, pack_toolbar=False)
    toolbar.update()
    toolbar.grid(row=0, column=0, sticky="ew")

    canvas_widget = canvas.get_tk_widget()
    canvas_widget.grid(row=1, column=0, sticky="nsew")

    frame_plot.grid_rowconfigure(0, weight=0)
    frame_plot.grid_rowconfigure(1, weight=1)
    frame_plot.grid_columnconfigure(0, weight=1)

    # Remember current figure, then close it from pyplot/Jupyter management.
    current_fig = fig
    plt.close(fig)


def create_empty_figure():
    fig = Figure(figsize=(6, 4), dpi=100)
    ax = fig.add_subplot(111)
    ax.set_axis_off()
    return fig


def plot_model():
    global RandTiling

    try:
        if RandTiling is None:
            raise ValueError("No model has been created yet.")

        start = time.perf_counter()

        RandTiling.plot(
            edge=var_edge.get(),
            paths=var_paths.get(),
            dots=var_dots.get(),
            coloring=var_coloring.get(),
            show_figure=False
        )

        fig = RandTiling.fig
        display_figure(fig)

        elapsed = time.perf_counter() - start
        write_log(f"Rendered plot in {elapsed:.3f}s.")

    except Exception as e:
        write_log(f"Could not render plot: {e}", level="ERROR")


button_plot_model = ttk.Button(
    frame_plot_options,
    text="Plot",
    command=plot_model
)
button_plot_model.grid(
    row=4,
    column=0,
    columnspan=2,
    sticky="ew",
    padx=8,
    pady=(8, 8)
)


# Set initial states
update_type_inputs()
update_plot_inputs()
update_coloring_options()

# Display an empty figure at startup, so the toolbar is already visible.
# It is also closed from pyplot/Jupyter management by display_figure().
display_figure(create_empty_figure())

write_log("Ready.")


# Layout resizing
frame_model.grid_columnconfigure(1, weight=1)
frame_plot.grid_rowconfigure(1, weight=1)
frame_plot.grid_columnconfigure(0, weight=1)

root.mainloop()

ImportError: Numba needs NumPy 2.4 or less. Got NumPy 2.5.

In [2]:
import matplotlib
#matplotlib.__version__

ModuleNotFoundError: No module named 'matplotlib.backends.registry'

# Nicer

In [1]:
import tkinter as tk
from tkinter import messagebox

import customtkinter as ctk

import numpy as np
import ast
import time
import io
import ctypes
from contextlib import redirect_stdout

import RandomTilings as RT

import matplotlib.pyplot as plt

from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.figure import Figure


# ---------------------------------------------------------------------
# Appearance
# ---------------------------------------------------------------------

ctk.set_appearance_mode("System")
ctk.set_default_color_theme("blue")


# ---------------------------------------------------------------------
# Windows taskbar icon support
# ---------------------------------------------------------------------

try:
    myappid = "randomtilingstudio.app.1"
    ctypes.windll.shell32.SetCurrentProcessExplicitAppUserModelID(myappid)
except Exception:
    pass


# ---------------------------------------------------------------------
# Main window
# ---------------------------------------------------------------------

root = ctk.CTk()
root.title("Random Tiling Studio")
root.geometry("1000x700")

try:
    root.iconbitmap("RTS logo.ico")
except Exception:
    pass


# ---------------------------------------------------------------------
# Helper for section-like frames
# ---------------------------------------------------------------------

def make_section(parent, title, row, column=0, **grid_kwargs):
    section = ctk.CTkFrame(parent, corner_radius=12)
    section.grid(row=row, column=column, **grid_kwargs)

    title_label = ctk.CTkLabel(
        section,
        text=title,
        font=ctk.CTkFont(size=15, weight="bold")
    )
    title_label.grid(
        row=0,
        column=0,
        columnspan=2,
        sticky="w",
        padx=12,
        pady=(10, 6)
    )

    return section, title_label


# ---------------------------------------------------------------------
# Layout
# ---------------------------------------------------------------------

frame_input_container = ctk.CTkFrame(root, corner_radius=0, width=320)
frame_input_container.grid(row=0, column=0, sticky="ns", padx=(10, 5), pady=10)
frame_input_container.grid_propagate(False)

frame_input = ctk.CTkScrollableFrame(
    frame_input_container,
    corner_radius=0
)
frame_input.grid(row=0, column=0, sticky="nsew")

frame_right = ctk.CTkFrame(root, corner_radius=0)
frame_right.grid(row=0, column=1, sticky="nsew", padx=(5, 10), pady=10)

frame_plot = ctk.CTkFrame(frame_right, corner_radius=12)
frame_plot.grid(row=0, column=0, sticky="nsew", padx=0, pady=(0, 8))

frame_log, label_log_title = make_section(
    frame_right,
    "Log",
    row=1,
    sticky="ew",
    padx=0,
    pady=(0, 0)
)

root.grid_rowconfigure(0, weight=1)
root.grid_columnconfigure(0, weight=0)
root.grid_columnconfigure(1, weight=1)

frame_input_container.grid_rowconfigure(0, weight=1)
frame_input_container.grid_columnconfigure(0, weight=1)
frame_input.grid_columnconfigure(0, weight=1)

frame_right.grid_rowconfigure(0, weight=1)
frame_right.grid_rowconfigure(1, weight=0)
frame_right.grid_columnconfigure(0, weight=1)


# ---------------------------------------------------------------------
# Variables
# ---------------------------------------------------------------------

current_theme_mode = ctk.get_appearance_mode()

if current_theme_mode == "Dark":
    var_theme_mode = tk.StringVar(value="Dark")
else:
    var_theme_mode = tk.StringVar(value="Light")

var_auto_plot = tk.BooleanVar(value=False)

var_type = tk.StringVar(value="Aztec")
var_n = tk.IntVar(value=1)

var_a = tk.DoubleVar(value=1.0)
var_b = tk.DoubleVar(value=1.0)
var_c = tk.DoubleVar(value=1.0)

var_edge = tk.DoubleVar(value=0.0)
var_paths = tk.BooleanVar(value=False)
var_dots = tk.BooleanVar(value=False)
var_coloring = tk.StringVar(value="standard")

weight_matrix = np.array([[1]], dtype=float)

RandTiling = None

canvas = None
toolbar = None
current_fig = None


# ---------------------------------------------------------------------
# Log window
# ---------------------------------------------------------------------

text_log = ctk.CTkTextbox(
    frame_log,
    height=110,
    wrap="word"
)
text_log.grid(
    row=1,
    column=0,
    columnspan=2,
    sticky="ew",
    padx=12,
    pady=(0, 8)
)
text_log.configure(state="disabled")

frame_log_controls = ctk.CTkFrame(
    frame_log,
    fg_color="transparent"
)
frame_log_controls.grid(
    row=2,
    column=0,
    columnspan=2,
    sticky="ew",
    padx=12,
    pady=(0, 12)
)

frame_log_controls.grid_columnconfigure(2, weight=1)


def update_theme_button_text():
    if var_theme_mode.get() == "Dark":
        button_theme_mode.configure(text="Light mode")
    else:
        button_theme_mode.configure(text="Dark mode")


def switch_theme_mode():
    if var_theme_mode.get() == "Dark":
        ctk.set_appearance_mode("Light")
        var_theme_mode.set("Light")
    else:
        ctk.set_appearance_mode("Dark")
        var_theme_mode.set("Dark")

    update_theme_button_text()


button_theme_mode = ctk.CTkButton(
    frame_log_controls,
    text="",
    width=110,
    command=switch_theme_mode
)
button_theme_mode.grid(
    row=0,
    column=0,
    sticky="w",
    padx=(0, 12)
)

check_auto_plot = ctk.CTkCheckBox(
    frame_log_controls,
    text="Auto plot",
    variable=var_auto_plot
)
check_auto_plot.grid(
    row=0,
    column=1,
    sticky="w",
    padx=(0, 12)
)

button_clear_log = ctk.CTkButton(
    frame_log_controls,
    text="Clear Log",
    width=100
)
button_clear_log.grid(
    row=0,
    column=3,
    sticky="e"
)

frame_log.grid_columnconfigure(0, weight=1)


def write_log(message, level="INFO"):
    text_log.configure(state="normal")
    text_log.insert("end", f"[{level}] {message}\n")
    text_log.see("end")
    text_log.configure(state="disabled")


def clear_log():
    text_log.configure(state="normal")
    text_log.delete("1.0", "end")
    text_log.configure(state="disabled")


button_clear_log.configure(command=clear_log)


# ---------------------------------------------------------------------
# Model frame
# ---------------------------------------------------------------------

frame_model, label_model_title = make_section(
    frame_input,
    "Model",
    row=0,
    sticky="ew",
    padx=0,
    pady=(0, 8)
)

frame_model.grid_columnconfigure(1, weight=1)


ctk.CTkLabel(frame_model, text="Type").grid(
    row=1,
    column=0,
    sticky="nw",
    padx=(12, 8),
    pady=6
)

frame_type = ctk.CTkFrame(frame_model, fg_color="transparent")
frame_type.grid(row=1, column=1, sticky="w", padx=(0, 12), pady=6)


def update_coloring_options():
    if "combo_coloring" not in globals():
        return

    if var_type.get() == "Aztec":
        values = ["standard", "alternative", "gray", "aztec gray"]
    else:
        values = ["standard", "alternative", "gray"]

    combo_coloring.configure(values=values)

    if var_coloring.get() not in values:
        var_coloring.set("standard")


def update_type_inputs():
    is_hexagon = var_type.get() == "Hexagon"

    if is_hexagon:
        entry_state = "normal"
        title_color = ("gray10", "gray90")
        label_color = ("gray10", "gray90")
        frame_fg_color = ("gray92", "gray17")
        frame_border_color = ("gray70", "gray35")
        entry_text_color = ("gray10", "gray90")
        entry_border_color = ("gray60", "gray45")
    else:
        entry_state = "disabled"
        title_color = ("gray55", "gray45")
        label_color = ("gray55", "gray45")
        frame_fg_color = ("gray88", "gray20")
        frame_border_color = ("gray80", "gray30")
        entry_text_color = ("gray55", "gray45")
        entry_border_color = ("gray75", "gray30")

    label_hexagon_title.configure(text_color=title_color)

    label_a.configure(text_color=label_color)
    label_b.configure(text_color=label_color)
    label_c.configure(text_color=label_color)

    frame_hexagon.configure(
        fg_color=frame_fg_color,
        border_color=frame_border_color,
        border_width=1
    )

    entry_a.configure(
        state=entry_state,
        text_color=entry_text_color,
        border_color=entry_border_color
    )
    entry_b.configure(
        state=entry_state,
        text_color=entry_text_color,
        border_color=entry_border_color
    )
    entry_c.configure(
        state=entry_state,
        text_color=entry_text_color,
        border_color=entry_border_color
    )

    update_coloring_options()


radio_aztec = ctk.CTkRadioButton(
    frame_type,
    text="Aztec",
    variable=var_type,
    value="Aztec",
    command=update_type_inputs
)
radio_aztec.grid(row=0, column=0, sticky="w", pady=(0, 4))

radio_hexagon = ctk.CTkRadioButton(
    frame_type,
    text="Hexagon",
    variable=var_type,
    value="Hexagon",
    command=update_type_inputs
)
radio_hexagon.grid(row=1, column=0, sticky="w")


ctk.CTkLabel(frame_model, text="Size").grid(
    row=2,
    column=0,
    sticky="w",
    padx=(12, 8),
    pady=6
)

entry_n = ctk.CTkEntry(
    frame_model,
    textvariable=var_n,
    width=120
)
entry_n.grid(row=2, column=1, sticky="ew", padx=(0, 12), pady=6)


ctk.CTkLabel(frame_model, text="Weight").grid(
    row=3,
    column=0,
    sticky="w",
    padx=(12, 8),
    pady=6
)

button_weight_edit = ctk.CTkButton(
    frame_model,
    text="Edit"
)
button_weight_edit.grid(row=3, column=1, sticky="ew", padx=(0, 12), pady=6)


frame_hexagon = ctk.CTkFrame(
    frame_model,
    corner_radius=10,
    border_width=1
)
frame_hexagon.grid(
    row=4,
    column=0,
    columnspan=2,
    sticky="ew",
    padx=12,
    pady=(8, 10)
)

label_hexagon_title = ctk.CTkLabel(
    frame_hexagon,
    text="Hexagon scaling",
    font=ctk.CTkFont(size=13, weight="bold")
)
label_hexagon_title.grid(
    row=0,
    column=0,
    columnspan=6,
    sticky="w",
    padx=10,
    pady=(8, 4)
)

label_a = ctk.CTkLabel(frame_hexagon, text="a")
label_a.grid(row=1, column=0, sticky="w", padx=(10, 4), pady=(5, 10))

entry_a = ctk.CTkEntry(frame_hexagon, textvariable=var_a, width=70)
entry_a.grid(row=1, column=1, sticky="ew", padx=(0, 8), pady=(5, 10))

label_b = ctk.CTkLabel(frame_hexagon, text="b")
label_b.grid(row=1, column=2, sticky="w", padx=(0, 4), pady=(5, 10))

entry_b = ctk.CTkEntry(frame_hexagon, textvariable=var_b, width=70)
entry_b.grid(row=1, column=3, sticky="ew", padx=(0, 8), pady=(5, 10))

label_c = ctk.CTkLabel(frame_hexagon, text="c")
label_c.grid(row=1, column=4, sticky="w", padx=(0, 4), pady=(5, 10))

entry_c = ctk.CTkEntry(frame_hexagon, textvariable=var_c, width=70)
entry_c.grid(row=1, column=5, sticky="ew", padx=(0, 10), pady=(5, 10))

frame_hexagon.grid_columnconfigure(1, weight=1)
frame_hexagon.grid_columnconfigure(3, weight=1)
frame_hexagon.grid_columnconfigure(5, weight=1)


# ---------------------------------------------------------------------
# Weight editor
# ---------------------------------------------------------------------

def open_weight_editor():
    global weight_matrix

    window = ctk.CTkToplevel(root)
    window.title("Edit weight matrix")
    window.geometry("540x340")
    window.transient(root)
    window.grab_set()

    label = ctk.CTkLabel(
        window,
        text="Enter a numeric matrix, e.g. [[1], [1]]"
    )
    label.grid(row=0, column=0, sticky="w", padx=14, pady=(14, 6))

    text_weight = ctk.CTkTextbox(window, height=190, wrap="word")
    text_weight.grid(row=1, column=0, sticky="nsew", padx=14, pady=6)

    text_weight.insert("1.0", repr(weight_matrix.tolist()))

    def apply_weight():
        global weight_matrix

        raw = text_weight.get("1.0", "end").strip()

        try:
            value = ast.literal_eval(raw)
            arr = np.array(value, dtype=float)

            if arr.ndim != 2:
                raise ValueError("Weight must be a 2D matrix.")

            if arr.size == 0:
                raise ValueError("Weight matrix must not be empty.")

            weight_matrix = arr
            write_log(f"Updated weight matrix to shape {arr.shape}.")
            window.destroy()

        except Exception as e:
            messagebox.showerror(
                "Invalid weight matrix",
                f"Could not parse weight matrix:\n\n{e}"
            )

    frame_buttons = ctk.CTkFrame(window, fg_color="transparent")
    frame_buttons.grid(row=2, column=0, sticky="ew", padx=14, pady=(8, 14))

    button_cancel = ctk.CTkButton(
        frame_buttons,
        text="Cancel",
        fg_color="gray50",
        hover_color="gray40",
        command=window.destroy
    )
    button_cancel.grid(row=0, column=0, sticky="e", padx=5)

    button_apply = ctk.CTkButton(
        frame_buttons,
        text="Apply",
        command=apply_weight
    )
    button_apply.grid(row=0, column=1, sticky="e", padx=5)

    frame_buttons.grid_columnconfigure(0, weight=1)

    window.grid_rowconfigure(1, weight=1)
    window.grid_columnconfigure(0, weight=1)


button_weight_edit.configure(command=open_weight_editor)


# ---------------------------------------------------------------------
# Matplotlib display
# ---------------------------------------------------------------------

def display_figure(fig):
    global canvas, toolbar, current_fig

    if current_fig is not None:
        plt.close(current_fig)
        current_fig = None

    if toolbar is not None:
        toolbar.destroy()
        toolbar = None

    if canvas is not None:
        canvas.get_tk_widget().destroy()
        canvas = None

    canvas = FigureCanvasTkAgg(fig, master=frame_plot)
    canvas.draw()

    toolbar = NavigationToolbar2Tk(canvas, frame_plot, pack_toolbar=False)
    toolbar.update()
    toolbar.grid(row=0, column=0, sticky="ew", padx=8, pady=(8, 0))

    canvas_widget = canvas.get_tk_widget()
    canvas_widget.grid(row=1, column=0, sticky="nsew", padx=8, pady=8)

    frame_plot.grid_rowconfigure(0, weight=0)
    frame_plot.grid_rowconfigure(1, weight=1)
    frame_plot.grid_columnconfigure(0, weight=1)

    current_fig = fig
    plt.close(fig)


def create_empty_figure():
    fig = Figure(figsize=(6, 4), dpi=100)
    ax = fig.add_subplot(111)
    ax.set_axis_off()
    return fig


def plot_model():
    global RandTiling

    try:
        if RandTiling is None:
            raise ValueError("No model has been created yet.")

        start = time.perf_counter()

        RandTiling.plot(
            edge=var_edge.get(),
            paths=var_paths.get(),
            dots=var_dots.get(),
            coloring=var_coloring.get(),
            show_figure=False
        )

        fig = RandTiling.fig
        display_figure(fig)

        elapsed = time.perf_counter() - start
        write_log(f"Rendered plot in {elapsed:.3f}s.")

    except Exception as e:
        write_log(f"Could not render plot: {e}", level="ERROR")


# ---------------------------------------------------------------------
# Model routines
# ---------------------------------------------------------------------

def create_model():
    global RandTiling

    try:
        tiling_type = var_type.get()
        n = var_n.get()
        w = weight_matrix

        if n <= 0:
            raise ValueError("Size must be positive.")

        buffer = io.StringIO()
        start = time.perf_counter()

        with redirect_stdout(buffer):
            if tiling_type == "Aztec":
                RandTiling = RT.Aztec(n, w)

            elif tiling_type == "Hexagon":
                a = var_a.get()
                b = var_b.get()
                c = var_c.get()

                RandTiling = RT.Hexagon(n, w, a, b, c)

            else:
                raise ValueError(f"Unknown tiling type: {tiling_type}")

        elapsed = time.perf_counter() - start
        printed_output = buffer.getvalue().strip()

        write_log(f"Created {tiling_type} model in {elapsed:.3f}s.")

        if printed_output:
            write_log(printed_output, level="WARNING")

    except Exception as e:
        write_log(f"Could not create model: {e}", level="ERROR")


def shuffle_model():
    global RandTiling

    try:
        if RandTiling is None:
            raise ValueError("No model has been created yet.")

        buffer = io.StringIO()
        start = time.perf_counter()

        with redirect_stdout(buffer):
            RandTiling.shuffle()

        elapsed = time.perf_counter() - start
        printed_output = buffer.getvalue().strip()

        write_log(f"Shuffled model in {elapsed:.3f}s.")

        if printed_output:
            write_log(printed_output, level="WARNING")

        if var_auto_plot.get():
            plot_model()

    except Exception as e:
        write_log(f"Could not shuffle model: {e}", level="ERROR")


frame_model_buttons = ctk.CTkFrame(frame_model, fg_color="transparent")
frame_model_buttons.grid(
    row=5,
    column=0,
    columnspan=2,
    sticky="ew",
    padx=12,
    pady=(6, 12)
)

frame_model_buttons.grid_columnconfigure(0, weight=1)
frame_model_buttons.grid_columnconfigure(1, weight=1)

button_create_model = ctk.CTkButton(
    frame_model_buttons,
    text="Create Model",
    command=create_model
)
button_create_model.grid(
    row=0,
    column=0,
    sticky="ew",
    padx=(0, 5)
)

button_shuffle_model = ctk.CTkButton(
    frame_model_buttons,
    text="Shuffle",
    command=shuffle_model
)
button_shuffle_model.grid(
    row=0,
    column=1,
    sticky="ew",
    padx=(5, 0)
)


# ---------------------------------------------------------------------
# Plot options frame
# ---------------------------------------------------------------------

frame_plot_options, label_plot_options_title = make_section(
    frame_input,
    "Plot",
    row=1,
    sticky="ew",
    padx=0,
    pady=(0, 8)
)

frame_plot_options.grid_columnconfigure(1, weight=1)


ctk.CTkLabel(frame_plot_options, text="Edge width").grid(
    row=1,
    column=0,
    sticky="w",
    padx=(12, 8),
    pady=6
)

entry_edge = ctk.CTkEntry(
    frame_plot_options,
    textvariable=var_edge,
    width=120
)
entry_edge.grid(row=1, column=1, sticky="ew", padx=(0, 12), pady=6)


def update_plot_inputs():
    if var_paths.get():
        check_dots.configure(state="normal")
    else:
        check_dots.configure(state="disabled")
        var_dots.set(False)


check_paths = ctk.CTkCheckBox(
    frame_plot_options,
    text="Show paths",
    variable=var_paths,
    command=update_plot_inputs
)
check_paths.grid(
    row=2,
    column=0,
    columnspan=2,
    sticky="w",
    padx=12,
    pady=6
)

check_dots = ctk.CTkCheckBox(
    frame_plot_options,
    text="Show dots",
    variable=var_dots
)
check_dots.grid(
    row=3,
    column=0,
    columnspan=2,
    sticky="w",
    padx=12,
    pady=6
)


ctk.CTkLabel(frame_plot_options, text="Coloring").grid(
    row=4,
    column=0,
    sticky="w",
    padx=(12, 8),
    pady=6
)

combo_coloring = ctk.CTkComboBox(
    frame_plot_options,
    variable=var_coloring,
    values=["standard", "alternative", "gray", "aztec gray"],
    state="readonly"
)
combo_coloring.grid(row=4, column=1, sticky="ew", padx=(0, 12), pady=6)


button_plot_model = ctk.CTkButton(
    frame_plot_options,
    text="Plot",
    command=plot_model
)
button_plot_model.grid(
    row=5,
    column=0,
    columnspan=2,
    sticky="ew",
    padx=12,
    pady=(10, 12)
)


# ---------------------------------------------------------------------
# Initial states
# ---------------------------------------------------------------------

update_type_inputs()
update_plot_inputs()
update_coloring_options()
update_theme_button_text()

display_figure(create_empty_figure())

write_log("Ready.")

root.mainloop()

In [5]:
import matplotlib

In [4]:
import matplotlib